In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import numpy as np

from clonalg.antibody.real_antibody import RealAntibody, RealAntibodyBuilder
from clonalg.model.clonalg_optimization import OptimizationClonalg
from clonalg.problems.problem import *
from clonalg.visualization.visualization import *

In [4]:
ackley = Ackley()
n_dims = 2
bounds = [(-5.0, 5.0)] * n_dims


def factory() -> RealAntibody:
    genes = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
    return (
        RealAntibodyBuilder()
        .with_genes(genes)
        .with_bounds(bounds)
        .with_cost_fn(ackley)
        .build()
    )


clonalg = OptimizationClonalg(
    population_size=20,
    clone_factor=0.2,
    n_generations=300,
    antibody_factory=factory,
    hypermutation_strategy="rank",
    rho=2.0,
    suppression_threshold=0.1,
)

memory = clonalg.run()
memory.sort(key=lambda ab: ab.affinity(None), reverse=True)

100%|██████████| 300/300 [00:02<00:00, 131.48it/s]


In [5]:
best = memory[0]
paths = list(map(lambda x: np.array(x.genes).reshape(1, 2), memory))

print(f"Best solution: x = {best.genes}")
print(f"f(x)         = {ackley(best.genes):.6f}")
print(f"affinity     = {best.affinity(None):.6f}")

Best solution: x = [ 0.005509  -0.0111789]
f(x)         = 0.039381
affinity     = 0.962111


In [6]:
plot_3d_surface_without_grid(ackley, bounds=(-5.5, 5.5), grid_size=100)
plot_contour_and_paths(ackley, paths, bounds=(-5.5, 5.5))